# Problem Sheet 1 — Exercises 1.1 and 1.2

This notebook follows the course example: h5py, fields and data dictionaries, and loops over file chunks using extend. Run the cells in order. Paths default to TNG JupyterLab; update both basePath values for local data.

Data files are not included. Results and figures are generated when the notebook is run. Dependencies: NumPy, Matplotlib, and h5py. Optional Gaussian smoothing requires SciPy. Neither illustris_python nor Astropy is required.

Reference: https://www.tng-project.org/data/docs/specifications/



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import h5py


## 1.1 — The distribution of dark matter haloes

Load all halo masses and positions from TNG100-3-Dark at z=0. Plot one point per halo, then circular markers with area proportional to halo mass. Report the mass range in solar masses and compare maps with different minimum mass cuts or mass bins.

### 1.1.1 Paths and header


In [ ]:
basePath = '/home/tnguser/sims.TNG/TNG100-3-Dark/output'
snap = 99
filename = basePath + '/groups_%03d/fof_subhalo_tab_%03d.%s.hdf5' % (snap, snap, 0)
with h5py.File(filename, 'r') as f:
    halo_header = dict(f['Header'].attrs)
nchunks = int(halo_header['NumFiles'])
h = halo_header['HubbleParam']
a = halo_header['Time']
assert np.isclose(halo_header['Redshift'], 0)
print('Chunks:', nchunks, 'Redshift:', halo_header['Redshift'])


### 1.1.2 Load all file chunks using the course example


In [ ]:
fields = ['GroupMass', 'GroupPos']
data = {field: [] for field in fields}
for num in range(nchunks):
    filename = basePath + '/groups_%03d/fof_subhalo_tab_%03d.%s.hdf5' % (snap, snap, num)
    with h5py.File(filename, 'r') as f:
        if f['Header'].attrs['Ngroups_ThisFile'] == 0:
            continue
        for field in fields:
            data[field].extend(np.array(f['Group'][field][:]))
for field in fields:
    data[field] = np.array(data[field])
assert len(data['GroupMass']) == halo_header['Ngroups_Total']


### 1.1.3 Unit conversion and halo mass range

Multiply masses by $10^{10}/h$ to obtain $M_\odot$. Multiply coordinates by $a/(1000h)$ to obtain physical Mpc. At $z=0$, $a=1$.


In [ ]:
halo_mass = data['GroupMass'].astype(float) * 1e10 / h
halo_coords = data['GroupPos'].astype(float) * a / h / 1000
halo_boxsize = halo_header['BoxSize'] * a / h / 1000
halo_x = halo_coords[:, 0]
halo_y = halo_coords[:, 1]
print(f'Number of haloes: {len(halo_mass)}')
print(f'Mass range: {halo_mass.min():.3e} to {halo_mass.max():.3e} Msun')
print(f'Box size: {halo_boxsize:.3f} Mpc')


### 1.1.4 One point per halo, projected along the z axis


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(halo_x, halo_y, s=1, color='black', linewidths=0, rasterized=True)
ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]', xlim=(0, halo_boxsize),
       ylim=(0, halo_boxsize), title='TNG100-3-Dark: haloes at z=0')
ax.set_aspect('equal')
plt.show()


### 1.1.5 Circular marker area proportional to halo mass

The scatter parameter s specifies marker area in points squared. Markers do not represent physical halo radii.


In [ ]:
size = 500 * halo_mass / halo_mass.max()
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(halo_x, halo_y, s=size, marker='o', color='steelblue',
           alpha=0.6, linewidths=0, rasterized=True)
ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]', xlim=(0, halo_boxsize),
       ylim=(0, halo_boxsize), title='Marker area proportional to halo mass')
ax.set_aspect('equal')
plt.show()


### 1.1.6 Different minimum halo mass cuts


In [ ]:
mass_cuts = [1e11, 1e12, 1e13]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, cut in zip(axes, mass_cuts):
    select = halo_mass >= cut
    ax.scatter(halo_x[select], halo_y[select], s=4, color='black', linewidths=0)
    ax.set(xlabel='x [Mpc]', xlim=(0, halo_boxsize), ylim=(0, halo_boxsize),
           title=f'M ≥ {cut:.0e} Msun; N = {select.sum()}')
    ax.set_aspect('equal')
axes[0].set_ylabel('y [Mpc]')
plt.show()


### 1.1.7 Optional: non-overlapping halo mass bins


In [ ]:
mass_bins = [1e11, 1e12, 1e13, 1e14]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, low, high in zip(axes, mass_bins[:-1], mass_bins[1:]):
    select = (halo_mass >= low) & (halo_mass < high)
    ax.scatter(halo_x[select], halo_y[select], s=4, color='steelblue', linewidths=0)
    ax.set(xlabel='x [Mpc]', xlim=(0, halo_boxsize), ylim=(0, halo_boxsize),
           title=f'{low:.0e} ≤ M < {high:.0e} Msun; N = {select.sum()}')
    ax.set_aspect('equal')
axes[0].set_ylabel('y [Mpc]')
plt.show()


### 1.1 Discussion — complete after running

We use GroupMass and GroupPos from the FoF catalogue and project the full box onto the x-y plane. The marker area in the mass-weighted map is proportional to halo mass.

The halo masses range from **[minimum]** to **[maximum]** Msun. The spatial distribution shows **[describe the actual maps]**. After increasing the mass threshold, **[describe the changes]**.


## 1.2 — The distribution of dark matter particles

Load the dark matter particle masses and coordinates from TNG50-4-Dark at z=0. Plot every particle and make 2D histograms with different pixel sizes. Convert the maps to surface density in Msun/Mpc² and describe their structure. Gaussian smoothing is optional.

### 1.2.1 Paths and snapshot header

All dark matter particles have the same mass, read from MassTable[1].


In [ ]:
basePath = '/home/tnguser/sims.TNG/TNG50-4-Dark/output'
snap = 99
filename = basePath + '/snapdir_%03d/snap_%03d.%s.hdf5' % (snap, snap, 0)
with h5py.File(filename, 'r') as f:
    particle_header = dict(f['Header'].attrs)
nchunks = int(particle_header['NumFilesPerSnapshot'])
h = particle_header['HubbleParam']
a = particle_header['Time']
assert np.isclose(particle_header['Redshift'], 0)
particle_boxsize = particle_header['BoxSize'] * a / h / 1000
particle_mass = particle_header['MassTable'][1] * 1e10 / h
print('Chunks:', nchunks, 'Redshift:', particle_header['Redshift'])
print(f'Particle mass: {particle_mass:.3e} Msun')
print(f'Box size: {particle_boxsize:.3f} Mpc')


### 1.2.2 Load all particles using the course example

The extend syntax is retained from the example. The large number of row objects requires substantial memory, and this cell may take some time.


In [ ]:
fields = ['Coordinates']
data = {field: [] for field in fields}
for num in range(nchunks):
    filename = basePath + '/snapdir_%03d/snap_%03d.%s.hdf5' % (snap, snap, num)
    with h5py.File(filename, 'r') as f:
        if f['Header'].attrs['NumPart_ThisFile'][1] == 0:
            continue
        for field in fields:
            data[field].extend(np.array(f['PartType1'][field][:]))
    print(f'Loaded chunk {num + 1}/{nchunks}')
for field in fields:
    data[field] = np.array(data[field])
print('Coordinates shape:', data['Coordinates'].shape)


### 1.2.3 Unit conversion and particle count verification


In [ ]:
Coordinates = data['Coordinates'] * (a / h / 1000)
x_axis = Coordinates[:, 0]
y_axis = Coordinates[:, 1]
nparticles = len(Coordinates)
expected = (int(particle_header['NumPart_Total'][1])
            + (int(particle_header['NumPart_Total_HighWord'][1]) << 32))
assert nparticles == expected, 'Incomplete snapshot'
print('Number of particles:', nparticles)
print(f'Total mass: {nparticles * particle_mass:.3e} Msun')


### 1.2.4 One point per particle (all particles; rendering may be slow)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(x=x_axis, y=y_axis, s=0.01, color='black', linewidths=0, rasterized=True)
ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]', xlim=(0, particle_boxsize),
       ylim=(0, particle_boxsize), title='TNG50-4-Dark: particles at z=0')
ax.set_aspect('equal')
plt.show()


### 1.2.5 Two-dimensional count histograms with different pixel sizes

Each pixel contains a particle count. The maps have independent color scales and different pixel areas; colors alone cannot be used to compare physical surface densities.


In [ ]:
nbins_list = [128, 256, 512]
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for ax, nbins in zip(axes, nbins_list):
    counts, xedges, yedges, image = ax.hist2d(
        x=x_axis, y=y_axis, weights=None, bins=nbins,
        range=[[0, particle_boxsize], [0, particle_boxsize]],
        norm=mpl.colors.LogNorm(), cmap='magma')
    ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]',
           title=f'{nbins} × {nbins}; pixel = {particle_boxsize/nbins:.3f} Mpc')
    ax.set_aspect('equal')
    fig.colorbar(image, ax=ax, label='Particles per pixel')
plt.show()


### 1.2.6 Surface density map

$\Sigma_{ij}=N_{ij}m_{\rm DM}/A_{\rm pixel}$, where $A_{\rm pixel}=(L/N_{\rm bins})^2$.

Assign each particle a weight of $m_{\rm DM}/A_{\rm pixel}$. The result is the surface density projected through the full depth of the box, in $M_\odot/\mathrm{Mpc}^2$.


In [ ]:
nbins = 256
pixel_size = particle_boxsize / nbins
pixel_area = pixel_size**2
weights = np.full(nparticles, particle_mass / pixel_area)
fig, ax = plt.subplots(figsize=(9, 8))
Sigma, xedges, yedges, image = ax.hist2d(
    x=x_axis, y=y_axis, weights=weights, bins=nbins,
    range=[[0, particle_boxsize], [0, particle_boxsize]],
    norm=mpl.colors.LogNorm(), cmap='magma')
ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]', title='Projected dark matter surface density')
ax.set_aspect('equal')
fig.colorbar(image, ax=ax, label=r'$\Sigma$ [$M_\odot$ / Mpc$^2$]')
plt.show()
del weights


### 1.2.7 Mass conservation check


In [ ]:
mass_from_map = Sigma.sum() * pixel_area
mass_from_particles = nparticles * particle_mass
print(f'Mass from map:       {mass_from_map:.6e} Msun')
print(f'Mass from particles: {mass_from_particles:.6e} Msun')
assert np.isclose(mass_from_map, mass_from_particles), 'Mass conservation failed'
print('Mass conservation passed')
print(f'Mean surface density: {Sigma.mean():.3e} Msun/Mpc^2')


### 1.2.8 Optional: Gaussian smoothing

sigma=1 corresponds to a standard deviation of one pixel. Use mode='wrap' for the periodic simulation box.


In [ ]:
from scipy.ndimage import gaussian_filter
sigma_pixels = 1.0
Sigma_smooth = gaussian_filter(Sigma, sigma=sigma_pixels, mode='wrap')
fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(
    np.ma.masked_less_equal(Sigma_smooth.T, 0), origin='lower',
    extent=[0, particle_boxsize, 0, particle_boxsize],
    norm=mpl.colors.LogNorm(), cmap='magma')
ax.set(xlabel='x [Mpc]', ylabel='y [Mpc]',
       title=f'Gaussian smoothing: sigma = {sigma_pixels * pixel_size:.3f} Mpc')
ax.set_aspect('equal')
fig.colorbar(image, ax=ax, label=r'$\Sigma$ [$M_\odot$ / Mpc$^2$]')
plt.show()


### 1.2 Discussion — complete after running

We project all dark matter particles onto the x-y plane. In the scatter plot, **[describe visibility and overlap]**. In the histogram maps, **[describe the structures actually visible]**.

Smaller pixels provide finer spatial sampling but contain fewer particles and are more sensitive to particle-counting noise. The surface density is the mass per pixel divided by its area and includes the full depth of the simulation box. The total mass recovered from the map agrees with the total particle mass.

Optional extensions from the problem sheet: overlay the TNG50-4-Dark halo distribution, or compare particle maps at different redshifts. These extensions are not executed here.
